Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

In [65]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

In [66]:
base_dir = Path.cwd()
# ---- load ----
df2 = pd.read_csv(base_dir/"data"/"train.csv")
df3 = pd.read_csv(base_dir/"data"/"score.csv")

print("train:", df2.shape)
print("score:", df3.shape)

train: (15075, 31)
score: (3000, 28)


In [67]:
# Missing Value Test
#df2 = pd.read_csv(base_dir/"data"/"train.csv")
print("Train shape:", df2.shape)
print("\nMissing values:")
print(df2.isnull().sum()[df2.isnull().sum() > 0])

print("\nTarget distribution (claim_count):")
print(df2['claim_count'].value_counts(dropna=False))

Train shape: (15075, 31)

Missing values:
driver_avg_age                  603
prior_year_mileage_000          903
prior_loss_amount              1208
payment_frequency               452
risk_score_external             752
late_payment_count             1053
first_claim_reported_date     13214
days_to_first_claim_report    13214
dtype: int64

Target distribution (claim_count):
claim_count
0    13214
1     1623
2      208
3       25
4        3
5        1
7        1
Name: count, dtype: int64


In [68]:
#---Create 2 new training files based on ideas for fixing missing data---

# 1. Track empty cells (evaluation)
# 2. Fix data according to rules
# 3. Handle categorical dirty strings (like business_type capitalization) as a bonus to make sure the data is actually fixed and ready
# 4. Save fixed_train.csv
# 5. Create train_pruned.csv by dropping rows that had edits made to them (excluding first_claim_reported_date and days_to_first_claim_report)

data_edits_dir = base_dir / "data_edits"
data_edits_dir.mkdir(parents=True, exist_ok=True)

# First, calculate the overall mean of prior_loss_amount for rows that actually have prior claims (prior_apd_claim_count > 0 or prior_al_claim_count > 0).
mean_prior_loss_with_claims = df2.loc[
    (df2['prior_loss_amount'].notnull()) & 
    ((df2['prior_apd_claim_count'] > 0) | (df2['prior_al_claim_count'] > 0)), 
    'prior_loss_amount'
].mean()

#print("Mean prior loss with claims:", mean_prior_loss_with_claims)

# Evaluation tracking function
def track_empty_cells(df):
    missing_counts = df.isnull().sum()
    return missing_counts[missing_counts > 0]

#print("--- INITIAL EMPTY CELLS ---")
#print(track_empty_cells(df2))

# Copy data so we don't edit original
df_fixed = df2.copy()

# Fix string capitalization in business_type first (critical for encoding later!)
df_fixed['business_type'] = df_fixed['business_type'].str.strip().str.upper()

# 1. Continuous feature averages
avg_driver_age = df_fixed['driver_avg_age'].mean()
avg_years_biz = df_fixed['years_in_business'].mean()
avg_risk_ext = df_fixed['risk_score_external'].mean()

# Track which rows are edited (excluding the leak variables)
edited_mask = pd.Series(False, index=df_fixed.index)

# Record edits
edited_mask |= df_fixed['driver_avg_age'].isnull()
df_fixed['driver_avg_age'] = df_fixed['driver_avg_age'].fillna(avg_driver_age)

edited_mask |= df_fixed['years_in_business'].isnull()
df_fixed['years_in_business'] = df_fixed['years_in_business'].fillna(avg_years_biz)

edited_mask |= df_fixed['risk_score_external'].isnull()
df_fixed['risk_score_external'] = df_fixed['risk_score_external'].fillna(avg_risk_ext)

# Fix prior_loss_amount
# Rule: Use 0 if there are no prior claims written. If there are prior claims, replace with the average loss.
prior_claims_exist = (df_fixed['prior_apd_claim_count'] > 0) | (df_fixed['prior_al_claim_count'] > 0)

# Conditions for nulls
null_prior_loss = df_fixed['prior_loss_amount'].isnull()

# Case A: No prior claims -> 0
df_fixed.loc[null_prior_loss & ~prior_claims_exist, 'prior_loss_amount'] = 0.0

# Case B: Prior claims exist -> average of non-null prior loss where claims exist
df_fixed.loc[null_prior_loss & prior_claims_exist, 'prior_loss_amount'] = mean_prior_loss_with_claims

edited_mask |= null_prior_loss

# For payment_frequency, let's treat mode or keep as missing? If we impute it, it's an edit. Let's fill missing payment_frequency with 'Annual' or 'Unknown'
# Let's say we fill with mode 'Annual' and track it.
mode_pay = df_fixed['payment_frequency'].mode()[0]
edited_mask |= df_fixed['payment_frequency'].isnull()
df_fixed['payment_frequency'] = df_fixed['payment_frequency'].fillna(mode_pay)

# Let's fill prior_year_mileage_000 and late_payment_count with averages/medians too
avg_mileage = df_fixed['prior_year_mileage_000'].mean()
edited_mask |= df_fixed['prior_year_mileage_000'].isnull()
df_fixed['prior_year_mileage_000'] = df_fixed['prior_year_mileage_000'].fillna(avg_mileage)

avg_late_pay = df_fixed['late_payment_count'].mean()
edited_mask |= df_fixed['late_payment_count'].isnull()
df_fixed['late_payment_count'] = df_fixed['late_payment_count'].fillna(avg_late_pay)

# Push the edited rows to the bottom for fixed_train.csv
df_unedited = df_fixed[~edited_mask]
df_edited_rows = df_fixed[edited_mask]
fixed_train_reordered = pd.concat([df_unedited, df_edited_rows], ignore_index=True)

# Write to the new data_edits directory and create pruned version
fixed_train_reordered.to_csv(data_edits_dir / "fixed_train.csv", index=False)
train_pruned = df_unedited.copy()
train_pruned.to_csv(data_edits_dir / "train_pruned.csv", index=False)
print(f"\nCreated 'fixed_train.csv' in '{data_edits_dir.name}'. {len(df_edited_rows)} edited rows pushed to bottom.")
print(f"Created 'train_pruned.csv' in '{data_edits_dir.name}'.")

# Verify empty cells in fixed_train.csv (only the leak columns should remain empty)
print("\n--- EMPTY CELLS IN FIXED TRAIN ---")
print(track_empty_cells(fixed_train_reordered))


Created 'fixed_train.csv' in 'data_edits'. 4326 edited rows pushed to bottom.
Created 'train_pruned.csv' in 'data_edits'.

--- EMPTY CELLS IN FIXED TRAIN ---
first_claim_reported_date     13214
days_to_first_claim_report    13214
dtype: int64


In [69]:
# ---- preprocessing ----

def preprocess_datasets(df_train_raw, df_score_raw):
    """
    Cleans, encodes, and aligns training and scoring datasets consistently.
    Also handles target-leakage drops and imputes missing scoring features
    using training statistics to prevent predict-time NaNs.
    """
    # Create clean copies
    train_clean = df_train_raw.copy()
    score_clean = df_score_raw.copy()
    
    # Clean text formatting in categorical columns
    categorical_cols = ["coverage_type", "business_type", "state", "payment_frequency"]
    for col in categorical_cols:
        if col in train_clean.columns:
            train_clean[col] = train_clean[col].astype(str).str.strip().str.upper()
        if col in score_clean.columns:
            score_clean[col] = score_clean[col].astype(str).str.strip().str.upper()
            
    # Define columns to drop (leakage and non-predictive metadata)
    leak_cols = [
        "claim_paid_amount_current_period", 
        "days_to_first_claim_report", 
        "first_claim_reported_date",
        "claim_status_current_period",
        "total_loss_amount"
    ]
    ignore_cols = ["policy_id", "insured_id", "snapshot_date", "policy_effective_date", "policy_expiration_date"]
    
    # Handle Target Creation
    if "claim_count" in train_clean.columns:
        train_clean["target"] = (train_clean["claim_count"] > 0).astype(int)
        y_train = train_clean["target"]
    else:
        y_train = None
        
    # Get final predictive features from training
    all_drops = leak_cols + ignore_cols + ["claim_count", "target"]
    feature_cols = [col for col in train_clean.columns if col not in all_drops]
    
    X_train = train_clean[feature_cols].copy()
    
    # --- ALIGN & IMPUTE SCORE COLUMNS WITH TRAIN STATISTICS ---
    X_score = pd.DataFrame(index=score_clean.index)
    for col in feature_cols:
        if col in score_clean.columns:
            # First, bring over the raw data
            X_score[col] = score_clean[col]
            
            # If there are any missing values in the score set, impute them using train stats
            if X_score[col].isnull().any():
                if col in categorical_cols:
                    # Impute missing categoricals with the mode from train
                    train_mode = X_train[col].mode()[0] if not X_train[col].mode().empty else "UNKNOWN"
                    X_score[col] = X_score[col].fillna(train_mode)
                else:
                    # Impute missing numericals with the mean from train
                    train_mean = X_train[col].mean()
                    X_score[col] = X_score[col].fillna(train_mean)
        else:
            # If a training feature is completely missing in score.csv (e.g., 'has_safety_program')
            # we fill it with 0/False so the shape and names match perfectly.
            X_score[col] = 0
            
    # Ensure correct column order matches X_train exactly
    X_score = X_score[feature_cols]
    
    # Consistent Categorical Encoding
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
    X_score[categorical_cols] = encoder.transform(X_score[categorical_cols])
    
    return X_train, y_train, X_score

In [70]:
# Load 3 types of data and pass them to the preprocessing function
df_fixed_train = pd.read_csv(base_dir / "data_edits" / "fixed_train.csv")
X_train_fixed, y_train_fixed, X_score_fixed = preprocess_datasets(df_fixed_train, df3)

df_pruned_train = pd.read_csv(base_dir / "data_edits" / "train_pruned.csv")
X_train_pruned, y_train_pruned, X_score_pruned = preprocess_datasets(df_pruned_train, df3)

# Raw train data with missing entries kept intact!
X_train_sparse, y_train_sparse, X_score_sparse = preprocess_datasets(df2, df3)

In [71]:
# ---- target & dataset verification ---- We find out what percentage of the dataset has claims.
# 1. Base Class Rate
df2["target"] = (df2["claim_count"] > 0).astype(int)
print(f"Original Train Class Balance (Positive Rate): {df2['target'].mean():.3%}")

# 2. Pruned Class Rate (to check if dropping rows introduces bias)
df_pruned = pd.read_csv(base_dir / "data_edits" / "train_pruned.csv")
pruned_target = (df_pruned["claim_count"] > 0).astype(int)
print(f"Pruned Train Class Balance  (Positive Rate): {pruned_target.mean():.3%}")

# 3. Fixed Class Rate (reordered + artificial data)
df_fixed = pd.read_csv(base_dir / "data_edits" / "fixed_train.csv")
fixed_target = (df_fixed["claim_count"] > 0).astype(int)
print(f"Fixed Train Class Balance   (Positive Rate): {fixed_target.mean():.3%}")

Original Train Class Balance (Positive Rate): 12.345%
Pruned Train Class Balance  (Positive Rate): 12.308%
Fixed Train Class Balance   (Positive Rate): 12.345%


In [72]:
# ---- feature selection & experiment preparation ----

# Load original datasets
df_raw_train = pd.read_csv(base_dir / "data" / "train.csv")
df_score = pd.read_csv(base_dir / "data" / "score.csv")

# Load preprocessed datasets
df_pruned_train = pd.read_csv(base_dir / "data_edits" / "train_pruned.csv")
df_fixed_train = pd.read_csv(base_dir / "data_edits" / "fixed_train.csv")

# Experiment 1: Pruned Complete Data
X_pruned, y_pruned, X_score_pruned = preprocess_datasets(df_pruned_train, df_score)

# Experiment 2: Artificially Completed Data (Imputed)
X_fixed, y_fixed, X_score_fixed = preprocess_datasets(df_fixed_train, df_score)

# Experiment 3: Sparse Data (Missing data left intact for algorithms like LightGBM)
X_sparse, y_sparse, X_score_sparse = preprocess_datasets(df_raw_train, df_score)

# Quick validation check on shape uniformity 
print("Feature preparation complete!")
print(f" -> Pruned Experiment Features: {X_pruned.shape[1]} features, {X_pruned.shape[0]} samples.")
print(f" -> Imputed Experiment Features: {X_fixed.shape[1]} features, {X_fixed.shape[0]} samples.")
print(f" -> Sparse Experiment Features:  {X_sparse.shape[1]} features, {X_sparse.shape[0]} samples.")
print("\nFinal Predictive Feature Columns:")
print(list(X_fixed.columns))

Feature preparation complete!
 -> Pruned Experiment Features: 20 features, 10749 samples.
 -> Imputed Experiment Features: 20 features, 15075 samples.
 -> Sparse Experiment Features:  20 features, 15075 samples.

Final Predictive Feature Columns:
['coverage_type', 'vehicle_count', 'vehicle_avg_age', 'driver_count', 'driver_avg_age', 'years_in_business', 'prior_year_mileage_000', 'business_type', 'state', 'prior_apd_claim_count', 'prior_al_claim_count', 'prior_loss_amount', 'deductible', 'coverage_limit_000', 'annual_premium', 'payment_frequency', 'risk_score_external', 'has_safety_program', 'num_heavy_vehicles', 'late_payment_count']


In [73]:
# ---- fit model (temporal split & multi-model evaluation) ----
# 1. Define a function to perform a Temporal Split based on the original dates
def temporal_split_by_year(df_raw, X_processed, y_processed):
    """
    Splits the processed features and targets temporally.
    Train: 2020 & 2021 | Validation: 2022 (latest year of train.csv)
    """
    # Extract the year from policy_effective_date or snapshot_date
    years = pd.to_datetime(df_raw["policy_effective_date"], dayfirst=True).dt.year
    
    # Validation mask: 2022
    val_mask = (years == 2022)
    train_mask = ~val_mask
    
    X_tr = X_processed[train_mask]
    y_tr = y_processed[train_mask]
    X_val = X_processed[val_mask]
    y_val = y_processed[val_mask]
    
    return X_tr, X_val, y_tr, y_val

# Dictionary to hold our trained models for scoring later
trained_models = {}
evaluation_results = []

# List of experiments to run
experiments = [
    ("Pruned Complete Data", df_pruned_train, X_pruned, y_pruned),
    ("Artificially Completed Data", df_fixed_train, X_fixed, y_fixed),
    ("Sparse Data (NaNs Intact)", df_raw_train, X_sparse, y_sparse)
]

print("--- STARTING EXPERIMENTAL MODEL TRAINING LOOP ---\n")

for name, df_raw_source, X_all, y_all in experiments:
    # A. Perform the temporal split
    X_tr, X_val, y_tr, y_val = temporal_split_by_year(df_raw_source, X_all, y_all)
    
    print(f"Running Experiment: {name}")
    print(f" -> Train size: {X_tr.shape[0]} | Val size: {X_val.shape[0]}")
    
    # B. Define candidate models
    # Note: HistGradientBoosting natively handles missing values (perfect for Sparse Data!)
    models = {
        "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=150, max_depth=10, class_weight="balanced", random_state=42),
        "HistGradientBoosting": HistGradientBoostingClassifier(max_depth=6, random_state=42)
    }
    
    # C. Train and Evaluate
    for model_name, model in models.items():
        # Skip LogisticRegression/RandomForest for raw Sparse Data as they don't support NaNs out-of-the-box
        if name == "Sparse Data (NaNs Intact)" and model_name in ["LogisticRegression", "RandomForest"]:
            continue
            
        # Fit model
        model.fit(X_tr, y_tr)
        
        # Predict claim probabilities (underwriters care about risk probability, not 0 or 1!)
        val_preds_proba = model.predict_proba(X_val)[:, 1]
        
        # Calculate evaluation metrics
        auc = roc_auc_score(y_val, val_preds_proba)
        gini = 2 * auc - 1
        
        evaluation_results.append({
            "Data Paradigm": name,
            "Algorithm": model_name,
            "Validation ROC-AUC": round(auc, 4),
            "Validation Gini": round(gini, 4)
        })
        
        # Save the best performing configuration key to load for predictions
        trained_models[f"{name}_{model_name}"] = model

print("\n--- MODEL TRAINING COMPLETE ---\n")

# Display the comparison table
df_results = pd.DataFrame(evaluation_results)
print(df_results.sort_values(by="Validation Gini", ascending=False).to_string(index=False))

--- STARTING EXPERIMENTAL MODEL TRAINING LOOP ---

Running Experiment: Pruned Complete Data
 -> Train size: 8009 | Val size: 2740
Running Experiment: Artificially Completed Data
 -> Train size: 11302 | Val size: 3773
Running Experiment: Sparse Data (NaNs Intact)
 -> Train size: 11302 | Val size: 3773

--- MODEL TRAINING COMPLETE ---

              Data Paradigm            Algorithm  Validation ROC-AUC  Validation Gini
  Sparse Data (NaNs Intact) HistGradientBoosting              0.6687           0.3374
Artificially Completed Data         RandomForest              0.6679           0.3359
Artificially Completed Data HistGradientBoosting              0.6642           0.3285
       Pruned Complete Data         RandomForest              0.6587           0.3175
       Pruned Complete Data HistGradientBoosting              0.6320           0.2640
Artificially Completed Data   LogisticRegression              0.6071           0.2142
       Pruned Complete Data   LogisticRegression              

In [74]:
# ---- score new policies (multi-paradigm outputs) ----

# 1. Establish the output directory
outputs_dir = base_dir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

# 2. Define our scoring map: (Filename, Paradigm Name, Model Key, Feature Set)
scoring_configurations = [
    (
        "predictions_pruned.csv", 
        "Pruned Complete Data", 
        "Pruned Complete Data_RandomForest", 
        X_score_pruned
    ),
    (
        "predictions_imputed.csv", 
        "Artificially Completed Data", 
        "Artificially Completed Data_RandomForest", 
        X_score_fixed
    ),
    (
        "predictions_sparse.csv", 
        "Sparse Data (NaNs Intact)", 
        "Sparse Data (NaNs Intact)_HistGradientBoosting", 
        X_score_sparse
    )
]

print("--- GENERATING SCORING PREDICTIONS ---")

for filename, paradigm_name, model_key, X_score_features in scoring_configurations:
    # Retrieve the trained model from our dictionary
    model = trained_models.get(model_key)
    
    if model is not None:
        # Predict continuous risk probabilities (underwriters can use this for tier pricing!)
        # This extracts the probability of having a claim (class 1)
        risk_probabilities = model.predict_proba(X_score_features)[:, 1]
        
        # Create a clean DataFrame to output
        df_output = pd.DataFrame({
            "policy_id": df3["policy_id"],
            "predicted_claim_probability": np.round(risk_probabilities, 5)
        })
        
        # Save to /outputs directory without pandas auto-index
        output_filepath = outputs_dir / filename
        df_output.to_csv(output_filepath, index=False)
        
        # Print summary stats to confirm output health
        print(f"\n✓ Created '{filename}' ({paradigm_name})")
        print(f"  -> Path: {output_filepath}")
        print(f"  -> Mean predicted probability: {risk_probabilities.mean():.2%}")
        print(f"  -> Max predicted risk: {risk_probabilities.max():.2%}")
    else:
        print(f"✗ Could not find trained model for key: '{model_key}'")

print("\n--- ALL PREDICTION FILES WRITTEN SUCCESSFULLY ---")

--- GENERATING SCORING PREDICTIONS ---

✓ Created 'predictions_pruned.csv' (Pruned Complete Data)
  -> Path: c:\Users\Daniel\Documents\Case study zurich\Data Scientist Applied AI ML - Case Study\outputs\predictions_pruned.csv
  -> Mean predicted probability: 34.39%
  -> Max predicted risk: 72.13%

✓ Created 'predictions_imputed.csv' (Artificially Completed Data)
  -> Path: c:\Users\Daniel\Documents\Case study zurich\Data Scientist Applied AI ML - Case Study\outputs\predictions_imputed.csv
  -> Mean predicted probability: 36.06%
  -> Max predicted risk: 76.32%

✓ Created 'predictions_sparse.csv' (Sparse Data (NaNs Intact))
  -> Path: c:\Users\Daniel\Documents\Case study zurich\Data Scientist Applied AI ML - Case Study\outputs\predictions_sparse.csv
  -> Mean predicted probability: 14.90%
  -> Max predicted risk: 75.82%

--- ALL PREDICTION FILES WRITTEN SUCCESSFULLY ---


In [75]:
# ---- extract, align and export feature importances ----

# 1. Define the complete, exact list of all 31 columns in train.csv
original_train_columns = [
    "policy_id", "insured_id", "snapshot_date", "policy_effective_date", "policy_expiration_date",
    "coverage_type", "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000", "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount",
    "deductible", "coverage_limit_000", "annual_premium", "payment_frequency",
    "risk_score_external", "has_safety_program", "num_heavy_vehicles", "late_payment_count",
    "claim_count",
    "claim_paid_amount_current_period", "days_to_first_claim_report", "first_claim_reported_date",
    "claim_status_current_period", "total_loss_amount"
]

modeled_features = list(X_fixed.columns)
importance_data = {"Column Name": original_train_columns}

# 2. Define our winning configurations
winning_models = [
    ("Pruned Complete (RF)", "Pruned Complete Data_RandomForest"),
    ("Artificially Completed (RF)", "Artificially Completed Data_RandomForest"),
    ("Sparse Data (HGB)", "Sparse Data (NaNs Intact)_HistGradientBoosting")
]

for column_header, model_key in winning_models:
    model = trained_models.get(model_key)
    
    if model is not None:
        if "RandomForest" in model_key:
            # Random Forest uses native Gini impurity importance
            raw_importances = model.feature_importances_
            normalized_importances = (raw_importances / np.sum(raw_importances)) * 100.0
            
        elif "HistGradientBoosting" in model_key:
            print("Calculating permutation importance (using ROC AUC) for HistGradientBoosting (HGB)...")
            # Using ROC AUC makes the evaluation highly sensitive to changes in probability
            perm_importance = permutation_importance(
                model, 
                X_val, 
                y_val, 
                scoring='roc_auc',  # <-- This is the magic fix!
                n_repeats=5, 
                random_state=42
            )
            raw_importances = perm_importance.importances_mean
            
            # Clip negative importances (minor noise during permutation)
            raw_importances = np.clip(raw_importances, 0, None)
            
            # Normalize to 100%
            total_importance = np.sum(raw_importances)
            if total_importance > 0:
                normalized_importances = (raw_importances / total_importance) * 100.0
            else:
                normalized_importances = np.ones(len(modeled_features)) * (100.0 / len(modeled_features))
        
        # Map values to our 31 columns
        feature_to_weight = dict(zip(modeled_features, normalized_importances))
        col_weights = []
        for col in original_train_columns:
            if col in feature_to_weight:
                col_weights.append(f"{feature_to_weight[col]:.2f}%")
            else:
                col_weights.append("Irrelevant / NO")
                
        importance_data[column_header] = col_weights
    else:
        importance_data[column_header] = ["Model Not Found"] * len(original_train_columns)

# 3. Create DataFrame
df_importance_comparison = pd.DataFrame(importance_data)

# 4. Display the beautiful table
print("\n=== MODEL WEIGHTS COMPARISON TABLE (ALL 31 COLUMNS) ===")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(df_importance_comparison.to_string(index=False))

# 5. Export to outputs directory as a CSV file
outputs_dir = base_dir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
csv_filepath = outputs_dir / "model_feature_importances.csv"
df_importance_comparison.to_csv(csv_filepath, index=False)

print(f"\n✓ Successfully exported results as a CSV to: {csv_filepath}")

# Convert the dictionary to a DataFrame
output_filepath = outputs_dir / "model_feature_importances.csv"
df_importance_comparison.to_csv(output_filepath, index=False)

print(f"✓ Feature importance table exported to: {output_filepath}")

Calculating permutation importance (using ROC AUC) for HistGradientBoosting (HGB)...

=== MODEL WEIGHTS COMPARISON TABLE (ALL 31 COLUMNS) ===
                     Column Name Pruned Complete (RF) Artificially Completed (RF) Sparse Data (HGB)
                       policy_id      Irrelevant / NO             Irrelevant / NO   Irrelevant / NO
                      insured_id      Irrelevant / NO             Irrelevant / NO   Irrelevant / NO
                   snapshot_date      Irrelevant / NO             Irrelevant / NO   Irrelevant / NO
           policy_effective_date      Irrelevant / NO             Irrelevant / NO   Irrelevant / NO
          policy_expiration_date      Irrelevant / NO             Irrelevant / NO   Irrelevant / NO
                   coverage_type                4.31%                       3.96%            26.77%
                   vehicle_count                7.65%                       9.24%            30.17%
                 vehicle_avg_age                7.32%     